# Mattis EDA

Initial loading and first overview inspection for data in `data/raw/Alternative Medien/`.


In [112]:
from pathlib import Path
import pandas as pd

In [113]:
# Set Path
PROJECT_ROOT = Path.cwd().parent
BASE_DIR = PROJECT_ROOT / "data" / "raw" / "Alternative Medien"

print("cwd:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE_DIR:", BASE_DIR)
print("exists:", BASE_DIR.exists())

if not BASE_DIR.exists():
    raise FileNotFoundError(f"BASE_DIR not found: {BASE_DIR}")


cwd: /Users/MattisHaumann/Dev/Thesis/Initial EDA
PROJECT_ROOT: /Users/MattisHaumann/Dev/Thesis
BASE_DIR: /Users/MattisHaumann/Dev/Thesis/data/raw/Alternative Medien
exists: True


In [114]:
# CSV reader (for every source except RT)
def read_csv_resilient(csv_path: Path) -> pd.DataFrame:
    encodings = ("utf-8", "utf-8-sig", "latin-1")

    # normal read
    for enc in encodings:
        try:
            return pd.read_csv(
                csv_path,
                encoding=enc,
                low_memory=False,
                on_bad_lines="skip",
            )
        except Exception:
            pass

    # delimiter sniff (python engine)
    for enc in encodings:
        try:
            return pd.read_csv(
                csv_path,
                encoding=enc,
                sep=None,
                engine="python",
                low_memory=False,
                on_bad_lines="skip",
            )
        except Exception:
            pass

    raise ValueError(f"Could not read: {csv_path}")

In [115]:
dfs_by_source = {}
overview_rows = []

source_dirs = sorted(
    [p for p in BASE_DIR.iterdir() if p.is_dir()],
    key=lambda p: p.name.lower()
)

for source_dir in source_dirs:
    csv_files = sorted(source_dir.rglob("*.csv"))
    frames = []
    failed = 0

    for csv_file in csv_files:
        try:
            part = read_csv_resilient(csv_file)
            part["source"] = source_dir.name
            part["source_file"] = csv_file.name  # keep it simple
            frames.append(part)
        except Exception as e:
            failed += 1
            print(f"[WARN] {source_dir.name} -> {csv_file.name}: {e}")

    df_source = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    dfs_by_source[source_dir.name] = df_source

    overview_rows.append({
        "source": source_dir.name,
        "files_found": len(csv_files),
        "files_failed": failed,
        "total_articles": len(df_source),
        "n_columns": df_source.shape[1],
    })

# RT_de.xlsx at the root
rt_path = BASE_DIR / "RT_de.xlsx"
if rt_path.exists():
    df_rt = pd.read_excel(rt_path)
    df_rt["source"] = "RT_de"
    df_rt["source_file"] = rt_path.name
    dfs_by_source["RT_de"] = df_rt

    overview_rows.append({
        "source": "RT_de",
        "files_found": 1,
        "files_failed": 0,
        "total_articles": len(df_rt),
        "n_columns": df_rt.shape[1],
    })

overview_df = (
    pd.DataFrame(overview_rows)
    .sort_values(["total_articles", "source"], ascending=[False, True])
    .reset_index(drop=True)
)

display(overview_df)


,source,files_found,files_failed,total_articles,n_columns
0,deusch_pravda,496,0,397944,9
1,RT_de,1,0,12390,9
2,Nius_Rohdaten_neu,285,0,4885,29
3,apollo,165,0,3550,7
4,Tichy's Einblick,195,0,3126,8
5,Compact,214,0,2045,6
6,Antispiegel,307,0,914,9
7,Reitschuster,238,0,685,6


In [116]:
src = "deusch_pravda"

if src in dfs_by_source:
    df = dfs_by_source[src]

    if "time" in df.columns:
        # Parse German datetime and drop time part
        parsed = pd.to_datetime(
            df["time"],
            format="%d.%m.%Y, %H:%M",
            errors="coerce"
        )

        df["date"] = parsed.dt.normalize()  # keeps only YYYY-MM-DD

        dfs_by_source[src] = df

        print(f"[OK] {src}: created clean 'date' column (without time)")
    else:
        print(f"[WARN] {src}: no 'time' column found")


[OK] deusch_pravda: created clean 'date' column (without time)


In [117]:
DATE_CANDIDATES = ["Date", "date", "day", "published", "published_at", "datetime", "timestamp"]

date_rows = []

for source, df in dfs_by_source.items():
    date_min = pd.NaT
    date_max = pd.NaT

    for col in DATE_CANDIDATES:
        if col in df.columns:
            s = pd.to_datetime(df[col], errors="coerce").dt.normalize()
            if s.notna().any():
                date_min = s.min()
                date_max = s.max()
                break

    date_rows.append({
        "source": source,
        "date_min": date_min,
        "date_max": date_max,
    })

date_df = pd.DataFrame(date_rows)

overview_df = overview_df.drop(columns=["date_min", "date_max"], errors="ignore")
overview_df = overview_df.merge(date_df, on="source", how="left")

display(overview_df)


,source,files_found,files_failed,total_articles,n_columns,date_min,date_max
0,deusch_pravda,496,0,397944,9,2025-05-05,2025-09-02
1,RT_de,1,0,12390,9,2024-11-14,2026-01-19
2,Nius_Rohdaten_neu,285,0,4885,29,2025-05-01,2026-02-10
3,apollo,165,0,3550,7,2025-08-12,2026-02-10
4,Tichy's Einblick,195,0,3126,8,2025-08-01,2026-02-11
5,Compact,214,0,2045,6,2025-06-11,2026-02-10
6,Antispiegel,307,0,914,9,2025-04-10,2026-02-10
7,Reitschuster,238,0,685,6,2025-06-10,2026-02-10


# Articles per month

In [118]:
DATE_CANDIDATES = ["Date", "date", "day", "published", "published_at", "datetime", "timestamp"]

monthly_rows = []

for source, df in dfs_by_source.items():
    used_series = None

    # find a usable date column
    for col in DATE_CANDIDATES:
        if col in df.columns:
            s = pd.to_datetime(df[col], errors="coerce").dt.normalize()
            if s.notna().any():
                used_series = s
                break

    if used_series is None:
        continue

    # convert to year-month period
    ym = used_series.dt.to_period("M").astype(str)

    # count articles per month
    counts = ym.value_counts()

    for month, count in counts.items():
        monthly_rows.append({"source": source, "year_month": month, "articles": count})

monthly_df = pd.DataFrame(monthly_rows)

pivot = (
    monthly_df
    .pivot_table(index="source", columns="year_month", values="articles", aggfunc="sum", fill_value=0)
    .sort_index()
)

# optional: sort columns chronologically (YYYY-MM already sorts correctly as strings)
pivot = pivot.reindex(sorted(pivot.columns), axis=1)

display(pivot)


year_month,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02
source,,,,,,,,,,,,,,,,
Antispiegel,0,0,0,0,0,85,118,34,32,92,122,113,85,114,87,32
Compact,0,0,0,0,0,0,0,168,176,207,275,301,280,258,293,87
Nius_Rohdaten_neu,0,0,0,0,0,0,482,447,495,466,563,589,604,509,548,182
RT_de,522,1031,958,939,941,862,895,785,887,687,856,855,898,868,406,0
Reitschuster,0,0,0,0,0,0,0,65,106,72,100,0,88,0,67,22
Tichy's Einblick,0,0,0,0,0,0,0,0,0,458,486,510,483,504,505,180
apollo,0,0,0,0,0,0,0,0,0,176,565,647,643,635,646,238
deusch_pravda,0,0,0,0,0,0,22016,30688,39674,36574,2249,0,0,0,0,0


## Erste Insights
Klare Unterschiede zwischen den Quellen.

- **deusch_pravda** hat extrem hohe Zahlen, aber nur in wenigen Monaten (Mai–September 2025). Sind wahrscheinlich Telegramdaten und nicht klassisch news

- **RT_de** veröffentlicht konstant über einen langen Zeitraum (Ende 2024–Anfang 2026). Sehr strukturiert und institutionell

- **Nius, Apollo, Compact und Tichys Einblick** zeigen relativ regelmäßige monatliche Output-Zahlen. Auch stabile redaktionelle Routinen vermutlich

- **Antispiegel und Reitschuster** haben deutlich geringeres Volumen. Vermutlich kleinere Strukturen oder stärker meinungsgetriebene Formate.

**Fazit (erste EDA):**  
Schon auf dieser deskriptiven Ebene sieht man, dass die untersuchten „alternativen Medien“ sehr unterschiedlich arbeiten – sowohl im Volumen als auch in der zeitlichen Dynamik.


# Length of Articles

In [119]:
# map using lowercase source names to avoid case issues
TEXT_COLUMN_MAP = {
    "antispiegel": "Full_Text",
    "apollo": "Inhalt",
    "compact": "Inhalt",
    "deusch_pravda": "full_text",
    "reitschuster": "Inhalt",
    "nius_rohdaten_neu": "article_text",
    "rt_de": "Full_Text",
    "tichy's einblick": "article_text",
}

length_rows = []

for source, df in dfs_by_source.items():
    key = source.lower()  # normalize
    text_col = TEXT_COLUMN_MAP.get(key)

    if text_col is None or text_col not in df.columns:
        print(f"[WARN] {source}: text column not found or not mapped. (mapped={text_col})")
        continue

    words = df[text_col].astype(str).str.split().str.len()

    length_rows.append({
        "source": source,
        "articles": len(df),
        "text_col": text_col,
        "mean_words": words.mean(),
        "median_words": words.median(),
        "min_words": words.min(),
        "max_words": words.max(),
    })

length_df = (
    pd.DataFrame(length_rows)
    .sort_values("mean_words", ascending=False)
    .reset_index(drop=True)
)

display(length_df)

,source,articles,text_col,mean_words,median_words,min_words,max_words
0,Antispiegel,914,Full_Text,2203.440919,1720.0,170.0,18894.0
1,Reitschuster,685,Inhalt,1584.382482,1466.0,837.0,5014.0
2,Tichy's Einblick,3126,article_text,827.087332,765.0,42.0,5857.0
3,RT_de,12390,Full_Text,629.242696,470.5,7.0,4689.0
4,Nius_Rohdaten_neu,4885,article_text,586.183498,426.0,6.0,64721.0
5,Compact,2045,Inhalt,504.015159,417.0,23.0,4324.0
6,apollo,3550,Inhalt,444.925634,384.0,23.0,2662.0
7,deusch_pravda,397944,full_text,162.234063,114.0,1.0,7820.0


## Erste Insights 

Klare strukturelle Unterschiede zwischen den Quellen.

- **deusch_pravda** hat extrem hohe Volumina in kurzer Zeit. Gleichzeitig ist die durchschnittliche Textlänge deutlich niedriger. Das spricht für aggregierte Inhalte (z. B. Telegram) statt klassischer redaktioneller Langform.

- **RT_de** sowie **Antispiegel** und **Reitschuster** weisen im Schnitt deutlich längere Texte auf. Das deutet eher auf ausführlichere Beiträge oder kommentierende Formate hin.

- **Nius, Apollo, Compact und Tichys Einblick** liegen im mittleren Bereich. Hier sieht man eher typische Online-Artikel-Längen.

Wichtig:  
Die Textdaten sind teilweise noch stark „roh“. Es finden sich z. B. Werbeeinblendungen, wiederholte Textpassagen oder technische Artefakte. Vor tiefergehenden inhaltlichen Analysen (z. B. Sentiment, Topic Modeling) ist daher eine systematische Textbereinigung notwendig.

**Zwischenfazit:**  
Schon in dieser frühen EDA zeigen sich klare Unterschiede in Produktionslogik, Umfang und Struktur der Inhalte. Die Datenqualität und Textbereinigung werden dabei ein zentraler methodischer Schritt für die weitere Analyse sein.


## Dateninspektion pro Quelle

In [120]:
import random

SAMPLE_SIZE = 5

for source, df in dfs_by_source.items():
    print("\n" + "="*80)
    print(f"Source: {source}")
    print("="*80)

    if len(df) == 0:
        print("No data available.")
        continue

    sample_df = df.sample(min(SAMPLE_SIZE, len(df)), random_state=42)

    for i, row in sample_df.iterrows():
        print("\n--- Article ---")

        # Title if available
        if "title" in df.columns:
            print("Title:", row.get("title"))
        elif "Title" in df.columns:
            print("Title:", row.get("Title"))

        # Try mapped text column if already defined
        text_col = None
        for col in df.columns:
            if col.lower() in ["full_text", "text", "inhalt", "article_text"]:
                text_col = col
                break

        if text_col:
            text_preview = str(row[text_col])[:1000]
            print("\nText Preview:\n", text_preview)
        else:
            print("No obvious text column found.")



Source: Antispiegel

--- Article ---
Title: Beginnt jetzt die „GPS-Show“?

Text Preview:
 Propaganda
Beginnt jetzt die „GPS-Show“?
Schweden meldet eine große Zunahme der Störungen des GPS in der zivilen Luftfahrt. Schuld ist angeblich natürlich Russland und als Beleg erinnert der Spiegel an die Störung des Fluges von von der Leyen, die sich jedoch als Fake herausgestellt hat. Die westliche Propaganda lügt immer lustiger.
von Anti-Spiegel
6. September 2025 12:00 Uhr
Als am Montag gemeldet wurde, das GPS des Fluges von EU-Kommissionschefin Ursula von der Leyen von Warschau ins bulgarische Plowdiw sei von Russland gestört worden und die Maschine hätte deswegen eine Stunde lang Warteschleifen fliegen müssen, bevor die Piloten landen konnten, war schnell klar, dass die Geschichte frei erfunden war.
Die GPS-Lüge vom Montag
Erstens liegt Plowdiw etwa 200 Kilometer vom Schwarzen Meer entfernt, sodass, wenn Russland das GPS gestört hätte, das GPS im halben Bulgarien hätte gestört sein müssen. 

In [121]:
# Zeige das DataFrame für Russia Today (RT_de)
if "RT_de" in dfs_by_source:
    rt_df = dfs_by_source["RT_de"]
elif "df_rt" in globals():
    rt_df = df_rt
else:
    raise KeyError("RT_de nicht gefunden (weder in dfs_by_source noch als df_rt)")

print("RT_de shape:", rt_df.shape)
display(rt_df)

RT_de shape: (12390, 9)


Date                        Category  \
0      2026-01-19    Hauptseite\n/\nInternational   
1      2026-01-19    Hauptseite\n/\nInternational   
2      2026-01-19  Hauptseite\n/\nNahost-Konflikt   
3      2026-01-19          Hauptseite\n/\nSchweiz   
4      2026-01-19    Hauptseite\n/\nInternational   
...           ...                             ...   
12385  2024-11-14          Hauptseite\n/\nSchweiz   
12386  2024-11-14    Hauptseite\n/\nUkraine-Krieg   
12387  2024-11-14       Hauptseite\n/\nÖsterreich   
12388  2024-11-14      Hauptseite\n/\nDeutschland   
12389  2024-11-14      Hauptseite\n/\nDeutschland   

                                                                                                   Title  \
0                                Trump: Dänemark kann "russische Bedrohung" in Grönland nicht beseitigen   
1                                                  Mindestens 39 Tote bei schwerem Zugunglück in Spanien   
2              Syriens Machthaber al-Scharaa verschiebt wegen Militäroperationen seinen Termin in Berlin   
3                            Davos wird zum US-Forum: Zum Auftakt steht das WEF im Zeichen Donald Trumps   
4      Welch erbärmliche und prinzipienlose Kreaturen: Warum die Europäer nun Schutz bei Russland suchen   
...                                                                                                  ...   
12385                            Gedemütigt und geschlagen: Was Prostituierte in der Schweiz durchmachen   
12386                                    Bloomberg: Ukrainer spenden immer weniger für ihre Streitkräfte   
12387                                                Österreich: Traditionsmarke Kika/Leiner vor dem Aus   
12388                                                 Sachsen: CDU und SPD wollen jetzt allein koalieren   
12389                            Falls Mützenich Außenminister wird: Andrei Melnyk kündigt Selbstmord an   

                                                                                                                                                                                                                                                                                                                                                                 Summary  \
0                                                                                                           Die Nordatlantikallianz habe Dänemark 20 Jahre lang gebeten, die "russische Bedrohung" für Grönland zu beseitigen, aber Kopenhagen habe diese Aufgabe nicht bewältigt, schrieb US-Präsident Donald Trump. Jetzt sei es "an der Zeit", diese Frage zu klären.   
1                                                                                                      Bei einem schweren Zugunglück nahe Córdoba entgleisten und kollidierten zwei Hochgeschwindigkeitszüge. Mindestens 39 Menschen kamen ums Leben, rund 70 wurden verletzt. Die Ursache ist unklar. Der Bahnverkehr zwischen Madrid und Andalusien bleibt ausgesetzt.   
2                      Der für Anfang der Woche angekündigte Besuch des syrischen Übergangspräsidenten Ahmed al-Scharaa im Berliner Kanzleramt ist seitens Damaskus abgesagt worden. Grund seien die militärischen Ereignisse im Nordosten des Landes. Syrische Medien haben bekannt gegeben, dass mit den kurdischen SDF-Kräften ein Waffenstillstand vereinbart wurde.   
3                                                                                                                                                  Die größte US-Delegation aller Zeiten, massive Sicherheitsvorkehrungen und konkrete Machtpolitik machen Davos zur Bühne amerikanischer Interessen. Noch vor der Eröffnung steht fest: Dieses WEF gehört Donald Trump.   
4                                                                       Europa hat Russlands Rolle verkannt und zahlt nun den Preis, während Washington das Völkerrecht und die UNO offen entwertet. Der geplante "Friedensrat" der USA verstärkt die Angst vor ein

In [122]:
# RT today sample article + link to click

from IPython.display import HTML
from html import escape

pd.set_option("display.max_colwidth", None)

# get RT_de frame
if "RT_de" in dfs_by_source:
    rt = dfs_by_source["RT_de"]
elif "rt_df" in globals():
    rt = rt_df
else:
    raise KeyError("RT_de not found in dfs_by_source or as rt_df")

cols_map = {c.lower(): c for c in rt.columns}
full_col = cols_map.get("full_text") or cols_map.get("text") or cols_map.get("fulltext")
url_col  = cols_map.get("url") or cols_map.get("link")

if not full_col or not url_col:
    raise KeyError(f"Could not find text/url columns in RT_de. Available columns: {list(rt.columns)}")

sample_df = rt[[full_col, url_col]].sample(5, random_state=42).reset_index(drop=True)

def format_text(t):
    t = escape(str(t)).replace("\n", "<br>")
    return "<div style='max-width:1200px;white-space:normal;overflow:auto'>" + t + "</div>"

sample_df["full_text"] = sample_df[full_col].apply(format_text)
sample_df["url"] = sample_df[url_col].astype(str).apply(
    lambda u: f'<a href="{escape(u)}" target="_blank" rel="noopener noreferrer">{escape(u)}</a>'
)

html = sample_df[["full_text", "url"]].to_html(escape=False, index=False)
display(HTML(html))


## Topic Modelling (RT_de) — Preprocessing & Rationale

**Data input (RT_de)**
- Dataframe: `rt_topic_df` from `dfs_by_source["RT_de"]` (fallback: `rt_df`).
- Columns used (explicit): `Full_Text` for text and `Date` for timestamps.

**Why minimal preprocessing here?**
- `Full_Text` is already relatively clean, so we keep preprocessing simple and focused on stopword removal for better topic quality.

**Cleaning steps**
1. Copy raw model text into `topic_text_raw` (source columns are not overwritten).
2. Minimal normalization: lowercasing, URL removal, whitespace normalization.
3. Stopword removal: spaCy German stopwords (`spacy.lang.de.stop_words.STOP_WORDS`) + short custom list (`rt`, `https`, `http`, `www`, `de`, `freedert`, `online`).
4. Token filtering: remove numeric-only tokens and very short tokens (`< 2` chars).

**Chunking strategy (implemented)**
- We chunk each cleaned article into fixed token windows before BERTopic.
- Settings: `CHUNK_SIZE_TOKENS = 220`, `CHUNK_OVERLAP_TOKENS = 40`, `MIN_CHUNK_TOKENS = 40`.
- Why: long articles can dominate embeddings; chunking keeps topic signals more balanced and still preserves local context via overlap.

**Reproducibility notes**
- Deterministic rules and fixed embedding model: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.
- BERTopic settings fixed (`language="multilingual"`, `min_topic_size=25`, `calculate_probabilities=False`).
- Default uses full RT_de (`MAX_DOCS = None`); optional sampling uses `random_state=42`.

**Sanity checks**
- Print article/chunk counts and date range used for BERTopic.
- Show top tokens after cleaning to verify stopword removal worked (no `der/die/das` dominance).


In [123]:
%pip install spacy

Note: you may need to restart the kernel to use updated packages.


In [124]:
# BERTopic auf RT_de (clean + reproducible)
# Optional one-time install (if needed):
# %pip install bertopic sentence-transformers umap-learn hdbscan huggingface_hub spacy

import os
import re
from collections import Counter

import pandas as pd

try:
    from bertopic import BERTopic
except ImportError as e:
    raise ImportError(
        "BERTopic not installed. Run: %pip install bertopic sentence-transformers umap-learn hdbscan"
    ) from e

# Optional HF auth via environment only (no interactive prompt)
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    try:
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False, skip_if_logged_in=True)
        print('Hugging Face login via HF_TOKEN: active')
    except Exception as e:
        print(f'Hugging Face login skipped: {e}')
else:
    print('HF_TOKEN not set. Continuing without explicit HF login.')

# 1) Get RT_de dataframe from existing EDA objects
if 'RT_de' in dfs_by_source:
    rt_topic_df = dfs_by_source['RT_de'].copy()
elif 'rt_df' in globals():
    rt_topic_df = rt_df.copy()
else:
    raise KeyError('RT_de not found in dfs_by_source or rt_df')

# 2) Use explicit RT_de columns from your schema
required_cols = {'Full_Text', 'Date'}
missing_cols = required_cols.difference(rt_topic_df.columns)
if missing_cols:
    raise KeyError(f'Missing required RT_de columns: {sorted(missing_cols)}')

# 3) Create topic-modelling columns (keep original columns unchanged)
rt_topic_df['topic_text_raw'] = rt_topic_df['Full_Text'].astype(str)
rt_topic_df['topic_date'] = pd.to_datetime(rt_topic_df['Date'], errors='coerce')

# 4) Stopwords: spaCy German stopwords + short custom additions
try:
    from spacy.lang.de.stop_words import STOP_WORDS
except ImportError as e:
    raise ImportError(
        "spaCy German stopwords are required. Run: %pip install spacy"
    ) from e

german_stopwords = set(STOP_WORDS)
custom_stopwords = {'rt', 'https', 'http', 'www', 'de', 'freedert', 'online'}
all_stopwords = german_stopwords.union(custom_stopwords)

url_pattern = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)
token_pattern = re.compile(r'[^\W_]+', flags=re.UNICODE)

def clean_topic_text(text: str) -> str:
    # Minimal cleaning: keep content, mostly remove stopwords/noise
    text = str(text).lower()
    text = url_pattern.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = token_pattern.findall(text)
    kept = []
    for tok in tokens:
        if tok in all_stopwords:
            continue
        if tok.isnumeric():
            continue
        if len(tok) < 2:
            continue
        kept.append(tok)

    return ' '.join(kept)

rt_topic_df['topic_text_clean'] = rt_topic_df['topic_text_raw'].apply(clean_topic_text)

# 5) Chunking (implemented): split long cleaned texts into overlapping token windows
CHUNK_SIZE_TOKENS = 220
CHUNK_OVERLAP_TOKENS = 40
MIN_CHUNK_TOKENS = 40

if CHUNK_OVERLAP_TOKENS >= CHUNK_SIZE_TOKENS:
    raise ValueError('CHUNK_OVERLAP_TOKENS must be smaller than CHUNK_SIZE_TOKENS')

def chunk_text(text: str) -> list:
    tokens = str(text).split()
    if not tokens:
        return []

    step = CHUNK_SIZE_TOKENS - CHUNK_OVERLAP_TOKENS
    chunks = []
    for i in range(0, len(tokens), step):
        chunk_tokens = tokens[i:i + CHUNK_SIZE_TOKENS]
        if len(chunk_tokens) < MIN_CHUNK_TOKENS:
            continue
        chunks.append(' '.join(chunk_tokens))

    # If article is short but non-empty, keep one chunk
    if not chunks and len(tokens) > 0:
        chunks = [' '.join(tokens)]

    return chunks

rt_topic_df['topic_text_model'] = rt_topic_df['topic_text_clean']
rt_topic_df['topic_chunks'] = rt_topic_df['topic_text_model'].apply(chunk_text)

# Build chunk-level modelling frame for BERTopic
model_df = rt_topic_df[['topic_date', 'topic_chunks']].explode('topic_chunks').rename(columns={'topic_chunks': 'topic_text_model'})
model_df = model_df[
    model_df['topic_date'].notna()
    & model_df['topic_text_model'].notna()
    & model_df['topic_text_model'].str.len().gt(0)
].copy()

# Full RT_de by default; set MAX_DOCS for quick local tests only
MAX_DOCS = None
if MAX_DOCS is not None and len(model_df) > MAX_DOCS:
    model_df = model_df.sample(MAX_DOCS, random_state=42)

model_df = model_df.sort_values('topic_date').reset_index(drop=True)

if model_df.empty:
    raise ValueError('No usable RT_de chunks after preprocessing.')

docs = model_df['topic_text_model'].tolist()
timestamps = model_df['topic_date'].dt.to_pydatetime().tolist()

print(f'RT_de articles used: {rt_topic_df["topic_date"].notna().sum()}')
print(f'RT_de chunks used for BERTopic: {len(docs)}')
print(f'Date range: {model_df["topic_date"].min().date()} to {model_df["topic_date"].max().date()}')

# Sanity check: most frequent tokens after cleaning
token_counts = Counter(' '.join(rt_topic_df['topic_text_clean']).split())
top_tokens_df = pd.DataFrame(token_counts.most_common(20), columns=['token', 'count'])
display(top_tokens_df)

# 6) BERTopic configuration (stable + reproducible)
topic_model = BERTopic(
    language='multilingual',
    embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    min_topic_size=25,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs)
topic_info = topic_model.get_topic_info()

display(topic_info.head(20))
display(topic_model.visualize_barchart(top_n_topics=12))



HF_TOKEN not set. Continuing without explicit HF login.
RT_de articles used: 12390
RT_de chunks used for BERTopic: 25347
Date range: 2024-11-14 to 2026-01-19


,token,count
0,ukraine,28094
1,us,27029
2,russland,26916
3,trump,20534
4,usa,18611
5,eu,17386
6,russischen,14843
7,thema,14147
8,prozent,11785
9,präsident,10992


2026-02-19 19:56:03,398 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/793 [00:00<?, ?it/s]

2026-02-19 19:58:09,645 - BERTopic - Embedding - Completed ✓
2026-02-19 19:58:09,646 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 19:58:12,763 - BERTopic - Dimensionality - Completed ✓
2026-02-19 19:58:12,765 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 19:58:15,201 - BERTopic - Cluster - Completed ✓
2026-02-19 19:58:15,207 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 19:58:17,351 - BERTopic - Representation - Completed ✓


Topic  Count                                                Name  \
0      -1  12496                           -1_ukraine_russland_eu_us   
1       0   1147                               0_afd_spd_cdu_prozent   
2       1    847                     1_putin_ukraine_trump_präsident   
3       2    761                    2_israel_hamas_gaza_gazastreifen   
4       3    666                           3_venezuela_maduro_us_usa   
5       4    627                4_afd_deutschen_deutschland_deutsche   
6       5    596         5_selenskij_ukraine_selenskijs_ukrainischen   
7       6    459                             6_trump_musk_us_epstein   
8       7    452                 7_prozent_euro_deutschland_deutsche   
9       8    373                    8_china_chinesische_peking_zölle   
10      9    332          9_streitkräfte_ukrainischen_truppen_gebiet   
11     10    274                 10_europa_russland_europäischen_usa   
12     11    264  11_vermögenswerte_unternehmen_russischen_euroclear   
13     12    260            12_sanktionen_russland_eu_sanktionspaket   
14     13    234                       13_gas_gazprom_lng_kubikmeter   
15     14    234                     14_ukraine_milliarden_eu_dollar   
16     15    232              15_raketen_oreschnik_rakete_reichweite   
17     16    212                        16_corona_covid_who_pandemie   
18     17    195                     17_schiff_ostsee_schiffe_tanker   
19     18    182                      18_polizei_verletzt_täter_mann   

                                                                                                  Representation  \
0                              [ukraine, russland, eu, us, russischen, deutschland, usa, nato, thema, russische]   
1                                           [afd, spd, cdu, prozent, bsw, partei, grünen, habeck, fpö, parteien]   
2                       [putin, ukraine, trump, präsident, russland, treffen, moskau, wladimir, präsidenten, us]   
3     [israel, hamas, gaza, gazastreifen, israelischen, netanjahu, israels, israelische, geiseln, palästinenser]   
4             [venezuela, maduro, us, usa, venezuelas, venezolanischen, trump, karibik, venezolanische, caracas]   
5                              [afd, deutschen, deutschland, deutsche, berlin, krieg, russland, spd, cdu, thema]   
6            [selenskij, ukraine, selenskijs, ukrainischen, wladimir, ukrainische, kiew, trump, präsident, nabu]   
7                                 [trump, musk, us, epstein, trumps, donald, präsident, biden, elon, demokraten]   
8        [prozent, euro, deutschland, deutsche, deutschen, wirtschaft, milliarden, merz, millionen, unternehmen]   
9                                  [china, chinesische, peking, zölle, chinesischen, usa, xi, chinas, us, trump]   
10      [streitkräfte, ukrainischen, truppen, gebiet, stadt, sumy, pokrowsk, podoljaka, ukrainische, russischen]   
11                          [europa, russland, europäischen, usa, krieg, eu, nato, ukraine, russischen, staaten]   
12  [vermögenswerte, unternehmen, russischen, euroclear, eu, milliarden, belgien, eingefrorenen, russland, euro]   
13                [sanktionen, russland, eu, sanktionspaket, trump, us, moskau, russischen, verhängt, präsident]   
14               [gas, gazprom, lng, kubikmeter, pipeline, eu, russischem, gaslieferungen, slowakei, russisches]   
15                        [ukraine, milliarden, eu, dollar, euro, kiew, geld, us, vermögenswerte, unterstützung]   
16        [raketen, oreschnik, rakete, reichweite, drohnen, systeme, burewestnik, einsatz, starlink, satelliten]   
17                           [corona, covid, who, pandemie, mrna, impfstoffe, impfung, impfungen, epa, biontech]   
18                  [schiff, ostsee, schiffe, tanker, marine, schattenflotte, flagge, nato, estland, finnischen]   
19                       [polizei, verletzt, täter, mann, jährige, laut, messer, verletzte, magdeburg, jähriger]   

                                                      

In [125]:
# Topics over time (RT_de)
topics_over_time = topic_model.topics_over_time(
    docs=docs,
    timestamps=timestamps,
    nr_bins=20,
)

display(topics_over_time.head(20))

display(topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=10,
))



20it [00:45,  2.30s/it]


,Topic,Words,Frequency,Timestamp
0,-1,"ukraine, russland, nato, russischen, us",696,2024-11-13 13:39:21.600
1,0,"spd, bsw, afd, habeck, cdu",70,2024-11-13 13:39:21.600
2,1,"ukraine, putin, trump, russland, präsident",36,2024-11-13 13:39:21.600
3,2,"israel, israelischen, netanjahu, hamas, gaza",24,2024-11-13 13:39:21.600
4,3,"venezuela, peru, venezuelas, fentanyl, usa",6,2024-11-13 13:39:21.600
5,4,"krieg, deutschen, deutschland, deutsche, rothschild",33,2024-11-13 13:39:21.600
6,5,"selenskij, ukraine, selenskijs, trump, kiew",21,2024-11-13 13:39:21.600
7,6,"trump, biden, bhattacharya, us, hunter",15,2024-11-13 13:39:21.600
8,7,"prozent, deutschland, deutschen, ifo, euro",19,2024-11-13 13:39:21.600
9,8,"china, peking, chinesische, chinas, xi",27,2024-11-13 13:39:21.600


## Antispiegel — Topic Modelling (same pipeline as RT_de)


In [126]:
# Antispiegel topic modelling (same pipeline as RT_de)
# Uses the same cleaning, chunking, BERTopic setup, and outputs as RT_de.
if 'BERTopic' not in globals() or 'clean_topic_text' not in globals() or 'chunk_text' not in globals():
    raise RuntimeError('Please run the RT_de topic modelling cell first.')
antispiegel_source_key = next((k for k in dfs_by_source if k.lower() == 'antispiegel'), None)
if antispiegel_source_key is None:
    raise KeyError(f"Source not found for Antispiegel. Available sources: {list(dfs_by_source.keys())}")
antispiegel_topic_df = dfs_by_source[antispiegel_source_key].copy()
required_cols = {'Full Text', 'date'}
missing_cols = required_cols.difference(antispiegel_topic_df.columns)
if missing_cols:
    raise KeyError(f"Missing required columns for Antispiegel: {sorted(missing_cols)}")
# Keep only rows with available full text
antispiegel_topic_df = antispiegel_topic_df[antispiegel_topic_df['Full_Text'].notna()].copy()
# Same topic columns as RT_de
antispiegel_topic_df['topic_text_raw'] = antispiegel_topic_df['Full Text'].astype(str)
antispiegel_topic_df['topic_date'] = pd.to_datetime(antispiegel_topic_df['date'], errors='coerce')
# Same cleaning pipeline as RT_de
antispiegel_topic_df['topic_text_clean'] = antispiegel_topic_df['topic_text_raw'].apply(clean_topic_text)
antispiegel_topic_df['topic_text_model'] = antispiegel_topic_df['topic_text_clean']
# Same chunking pipeline as RT_de
antispiegel_topic_df['topic_chunks'] = antispiegel_topic_df['topic_text_model'].apply(chunk_text)
antispiegel_model_df = (
    antispiegel_topic_df[['topic_date', 'topic_chunks']]
    .explode('topic_chunks')
    .rename(columns={'topic_chunks': 'topic_text_model'})
)
antispiegel_model_df = antispiegel_model_df[
    antispiegel_model_df['topic_date'].notna()
    & antispiegel_model_df['topic_text_model'].notna()
    & antispiegel_model_df['topic_text_model'].str.len().gt(0)
].copy()
# Full corpus by default (no sampling)
MAX_DOCS = None
if MAX_DOCS is not None and len(antispiegel_model_df) > MAX_DOCS:
    antispiegel_model_df = antispiegel_model_df.sample(MAX_DOCS, random_state=42)
antispiegel_model_df = antispiegel_model_df.sort_values('topic_date').reset_index(drop=True)
if antispiegel_model_df.empty:
    raise ValueError('No usable chunks after preprocessing.')
antispiegel_docs = antispiegel_model_df['topic_text_model'].tolist()
antispiegel_timestamps = antispiegel_model_df['topic_date'].dt.to_pydatetime().tolist()
print(f'Antispiegel articles used: {antispiegel_topic_df["topic_date"].notna().sum()}')
print(f'Antispiegel chunks used for BERTopic: {len(antispiegel_docs)}')
print(f'Date range: {antispiegel_model_df["topic_date"].min().date()} to {antispiegel_model_df["topic_date"].max().date()}')
# Same sanity check as RT_de
antispiegel_token_counts = Counter(' '.join(antispiegel_topic_df['topic_text_clean']).split())
antispiegel_top_tokens_df = pd.DataFrame(antispiegel_token_counts.most_common(20), columns=['token', 'count'])
# Overall non-time topic overview (all documents/chunks)
display(antispiegel_top_tokens_df)
# Same BERTopic configuration as RT_de
antispiegel_topic_model = BERTopic(
    language='multilingual',
    embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    min_topic_size=25,
    calculate_probabilities=False,
    verbose=True,
)
antispiegel_topics, _ = antispiegel_topic_model.fit_transform(antispiegel_docs)
antispiegel_topic_info = antispiegel_topic_model.get_topic_info()
display(antispiegel_topic_info.head(20))
display(antispiegel_topic_model.visualize_barchart(top_n_topics=12))




Antispiegel articles used: 914
Antispiegel chunks used for BERTopic: 5493
Date range: 2025-04-10 to 2026-02-10


,token,count
0,russland,10035
1,ukraine,7533
2,antworten,6723
3,anmelden,6382
4,eu,6131
5,teilen,5612
6,usa,5424
7,trump,5041
8,spiegel,3900
9,mai,3856


2026-02-19 19:59:06,074 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/172 [00:00<?, ?it/s]

2026-02-19 19:59:45,504 - BERTopic - Embedding - Completed ✓
2026-02-19 19:59:45,507 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 19:59:47,202 - BERTopic - Dimensionality - Completed ✓
2026-02-19 19:59:47,203 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 19:59:47,336 - BERTopic - Cluster - Completed ✓
2026-02-19 19:59:47,338 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 19:59:47,845 - BERTopic - Representation - Completed ✓


Topic  Count                                   Name  \
0     -1    328         -1_russland_usa_spiegel_teilen   
1      0   4665  0_russland_ukraine_antworten_anmelden   
2      1    204        1_israel_gaza_israelische_hamas   
3      2    145      2_moldawien_wahlen_sandu_rumänien   
4      3     46               3_venezuela_usa_trump_us   
5      4     39    4_april_anmelden_antworten_menschen   
6      5     35         5_grönland_dänemark_usa_arktis   
7      6     31      6_tacheles_teilen_sendung_freitag   

                                                                                        Representation  \
0                           [russland, usa, spiegel, teilen, trump, anti, us, eu, antworten, anmelden]   
1                   [russland, ukraine, antworten, anmelden, eu, teilen, usa, trump, mai, deutschland]   
2     [israel, gaza, israelische, hamas, netanjahu, israelischen, israels, gazastreifen, teilen, iran]   
3  [moldawien, wahlen, sandu, rumänien, wahl, regierung, eu, parlamentswahlen, moldawischen, russland]   
4      [venezuela, usa, trump, us, maduro, öl, venezuelas, venezolanischen, venezolanische, regierung]   
5                 [april, anmelden, antworten, menschen, wasser, artikel, mal, fische, hormone, pille]   
6                      [grönland, dänemark, usa, arktis, grönlands, trump, kanada, us, nato, dänische]   
7                   [tacheles, teilen, sendung, freitag, stein, anti, lebt, osteuropa, beitrag, röper]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

18it [00:05,  3.16it/s]


,Topic,Words,Frequency,Timestamp
0,-1,"april, russland, antworten, anmelden, usa",30,2025-04-09 16:39:21.600
1,0,"april, antworten, anmelden, russland, ukraine",624,2025-04-09 16:39:21.600
2,1,"israel, türkei, teilen, iran, atomwaffen",2,2025-04-09 16:39:21.600
3,2,"moldawien, sandu, gagausien, kirche, regierung",10,2025-04-09 16:39:21.600
4,4,"april, anmelden, antworten, menschen, mal",26,2025-04-09 16:39:21.600
5,6,"tacheles, stein, teilen, sendung, robert",1,2025-04-09 16:39:21.600
6,-1,"pakistan, indien, antworten, anmelden, trump",28,2025-04-25 07:12:00.000
7,0,"antworten, anmelden, mai, russland, ukraine",477,2025-04-25 07:12:00.000
8,1,"israel, gaza, the, mai, of",11,2025-04-25 07:12:00.000
9,2,"moldawien, wahlen, sandu, rumänien, maia",19,2025-04-25 07:12:00.000


### Antispiegel — Dynamic Topic Modelling (over time)


In [ ]:
# Topics over time (same as RT_de)
antispiegel_topics_over_time = antispiegel_topic_model.topics_over_time(
    docs=antispiegel_docs,
    timestamps=antispiegel_timestamps,
    nr_bins=20,
)

display(antispiegel_topics_over_time.head(20))

display(antispiegel_topic_model.visualize_topics_over_time(
    antispiegel_topics_over_time,
    top_n_topics=10,
))


## Apollo — Topic Modelling (same pipeline as RT_de)


In [127]:
# Apollo topic modelling (same pipeline as RT_de)
# Uses the same cleaning, chunking, BERTopic setup, and outputs as RT_de.
if 'BERTopic' not in globals() or 'clean_topic_text' not in globals() or 'chunk_text' not in globals():
    raise RuntimeError('Please run the RT_de topic modelling cell first.')
apollo_source_key = next((k for k in dfs_by_source if k.lower() == 'apollo'), None)
if apollo_source_key is None:
    raise KeyError(f"Source not found for Apollo. Available sources: {list(dfs_by_source.keys())}")
apollo_topic_df = dfs_by_source[apollo_source_key].copy()
required_cols = {'Inhalt', 'Date'}
missing_cols = required_cols.difference(apollo_topic_df.columns)
if missing_cols:
    raise KeyError(f"Missing required columns for Apollo: {sorted(missing_cols)}")
# Keep only rows with available full text
apollo_topic_df = apollo_topic_df[apollo_topic_df['Inhalt'].notna()].copy()
# Same topic columns as RT_de
apollo_topic_df['topic_text_raw'] = apollo_topic_df['Inhalt'].astype(str)
apollo_topic_df['topic_date'] = pd.to_datetime(apollo_topic_df['Date'], errors='coerce')
# Same cleaning pipeline as RT_de
apollo_topic_df['topic_text_clean'] = apollo_topic_df['topic_text_raw'].apply(clean_topic_text)
apollo_topic_df['topic_text_model'] = apollo_topic_df['topic_text_clean']
# Same chunking pipeline as RT_de
apollo_topic_df['topic_chunks'] = apollo_topic_df['topic_text_model'].apply(chunk_text)
apollo_model_df = (
    apollo_topic_df[['topic_date', 'topic_chunks']]
    .explode('topic_chunks')
    .rename(columns={'topic_chunks': 'topic_text_model'})
)
apollo_model_df = apollo_model_df[
    apollo_model_df['topic_date'].notna()
    & apollo_model_df['topic_text_model'].notna()
    & apollo_model_df['topic_text_model'].str.len().gt(0)
].copy()
# Full corpus by default (no sampling)
MAX_DOCS = None
if MAX_DOCS is not None and len(apollo_model_df) > MAX_DOCS:
    apollo_model_df = apollo_model_df.sample(MAX_DOCS, random_state=42)
apollo_model_df = apollo_model_df.sort_values('topic_date').reset_index(drop=True)
if apollo_model_df.empty:
    raise ValueError('No usable chunks after preprocessing.')
apollo_docs = apollo_model_df['topic_text_model'].tolist()
apollo_timestamps = apollo_model_df['topic_date'].dt.to_pydatetime().tolist()
print(f'Apollo articles used: {apollo_topic_df["topic_date"].notna().sum()}')
print(f'Apollo chunks used for BERTopic: {len(apollo_docs)}')
print(f'Date range: {apollo_model_df["topic_date"].min().date()} to {apollo_model_df["topic_date"].max().date()}')
# Same sanity check as RT_de
apollo_token_counts = Counter(' '.join(apollo_topic_df['topic_text_clean']).split())
apollo_top_tokens_df = pd.DataFrame(apollo_token_counts.most_common(20), columns=['token', 'count'])
# Overall non-time topic overview (all documents/chunks)
display(apollo_top_tokens_df)
# Same BERTopic configuration as RT_de
apollo_topic_model = BERTopic(
    language='multilingual',
    embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    min_topic_size=25,
    calculate_probabilities=False,
    verbose=True,
)
apollo_topics, _ = apollo_topic_model.fit_transform(apollo_docs)
apollo_topic_info = apollo_topic_model.get_topic_info()
display(apollo_topic_info.head(20))
display(apollo_topic_model.visualize_barchart(top_n_topics=12))



Apollo articles used: 3550
Apollo chunks used for BERTopic: 5283
Date range: 2025-08-12 to 2026-02-10


,token,count
0,pay,7077
1,werbung,6967
2,news,4989
3,apollo,4899
4,prozent,4468
5,zahlen,4029
6,teilen,3739
7,unterstützen,3715
8,bank,3639
9,monatlich,3569


2026-02-19 19:59:54,955 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/166 [00:00<?, ?it/s]

2026-02-19 20:00:37,818 - BERTopic - Embedding - Completed ✓
2026-02-19 20:00:37,819 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 20:00:39,475 - BERTopic - Dimensionality - Completed ✓
2026-02-19 20:00:39,477 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 20:00:39,596 - BERTopic - Cluster - Completed ✓
2026-02-19 20:00:39,599 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 20:00:40,049 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,0,5067,0_werbung_pay_news_apollo,"[werbung, pay, news, apollo, prozent, afd, zahlen, teilen, unterstützen, kommentaren]","[monate wahl berliner abgeordnetenhaus verliert cdu regierenden bürgermeisters kai wegner spürbar rückhalt zeigt mittwoch veröffentlichte erhebung instituts infratest dimap sender rbb aktuellen zahlen schwarz rote koalition cdu spd mehrheit deutlich eingebüßt ergebnis sinkende werte beider regierungsparteien zurückgeht berlintrend sieht cdu derzeit prozent punkte juni punkte abgeordnetenhauswahl februar werbung unverändert platz liegt linkspartei prozent afd gewinnt hinzu erreicht prozent grünen punkt ebenfalls prozent steigen spd fällt prozent bündnis sahra wagenknecht verpasst prozent prozent hürde kleinere parteien prozent ergebnissen ergeben mehrere theoretische mehrheitskonstellationen weiterhin cdu geführte regierung einbindung grünen spd alternativ käme rot grün rotes bündnis führung linkspartei betracht werbung zusammenarbeit afd schließen parteien weiterhin cdu lehnt zudem bündnis linkspartei nächste wahl berliner abgeordnetenhaus findet september ha teilen kommentaren einmalig monatlich apollo news unterstützen zahlungsoptionen pay pay zahlen bank überweisung, afd erreicht mecklenburg vorpommern höchstwert laut umfrage infratest dimap auftrag ndr liegt partei prozent april insa erhebung nordkurier afd prozent gesehen zuwachs fast punkten wenigen monaten spd fällt aktuellen erhebung prozent landtagswahl prozent erzielt ergebnis halbieren schwäche profitiert afd cdu hingegen bleibt trotz oppositionsrolle prozent historischen tiefstand linke verbessert prozent bündnis sahra wagenknecht sinkt prozent grünen liegen prozent knapp sperrklausel werbung prozent befragten arbeit rot roten koalition unzufrieden prozent zufrieden ministerpräsidentin manuela schwesig schneidet vergleich prozent zufrieden prozent fortsetzung rot roten koalition rechnerisch bündnis spd cdu linken mehrheit erreichen regierung afd viererbündnis notwendig spd cdu linken grünen spd cdu linken bsw fällen müsste cdu linken kooperieren werbung mecklenburg vorpommern reiht serie ostdeutscher afd bestwerte sachsen anhalt gewählt lag partei zuletzt prozent thüringen prozent bundestagswahl februar afd nordosten klar stärkste kraft mecklenburg vorpommern erreichte prozent zweitstimmen gewann direktmandate cm teilen kommentaren einmalig monatlich apollo news unterstützen zahlungsoptionen pay pay zahlen bank überweisung, aktuellen erhebung meinungsforschungsinstituts insa bild sonntag bleibt afd prozent unverändert stärkste politische kraft deutschland union sonntagstrend prozentpunkt hinzugewinnen erreicht prozent spd verharrt prozent werte grünen linken bleiben stabil jeweils prozent fdp verliert punkt prozent bsw prozent unverändert bleibt sonstige parteien entfallen prozent stimmen delivered by ama setzt leichte aufwärtstrend union fort rangfolge parteien ändert afd liegt weiterhin knapp cdu csu parteipräferenzen fragte insa haltung sogenannten brandmauer afd klare mehrheit unionsanhänger steht kurs parteiführung positiv prozent wähler cdu csu begrüßen zusammenarbeit afd ausgeschlossen prozent unionsanhänger halten brandmauer falsch werbung gesamtbevölkerung finden laut insa umfrage bild sonntag prozent befragten brandmauer prozent ablehnen reihen union brandmauerkritische stimmen lauter gleichwohl betont merz afd hauptgegner kommenden wahlen partei wolle cdu erklärtermaßen zerstören warnte parteivorsitzende ha teilen kommentaren einmalig monatlich apollo news unterstützen zahlungsoptionen pay pay zahlen bank überweisung]"
1,1,61,1_pay_überweisung_zahlungsoptionen_bank,"[pay, überweisung, zahlungsoptionen, bank, einmalig, monatlich, unterstützen, teilen, kommentaren, zahlen]","[teilen kommentaren einmalig monatlich apollo news unterstützen zahlungsoptionen pay pay zahlen bank überweisung, teilen kommentaren einmalig monatlich apollo news unterstützen zahlungsoptionen pay pay zahlen ba

20it [00:06,  3.20it/s]


,Topic,Words,Frequency,Timestamp
0,0,"afd, pay, apollo, news, prozent",16,2025-08-11 19:37:55.200
1,0,"werbung, pay, prozent, news, apollo",219,2025-08-21 02:24:00.000
2,1,"pay, überweisung, zahlungsoptionen, bank, einmalig",3,2025-08-21 02:24:00.000
3,2,"pay, überweisung, zahlungsoptionen, bank, einmalig",3,2025-08-21 02:24:00.000
4,4,"pay, überweisung, zahlungsoptionen, bank, einmalig",3,2025-08-21 02:24:00.000
5,5,"pay, überweisung, zahlungsoptionen, bank, einmalig",3,2025-08-21 02:24:00.000
6,0,"pay, werbung, prozent, apollo, news",212,2025-08-30 04:48:00.000
7,1,"pay, überweisung, zahlungsoptionen, bank, einmalig",1,2025-08-30 04:48:00.000
8,2,"pay, überweisung, zahlungsoptionen, bank, einmalig",5,2025-08-30 04:48:00.000
9,4,"pay, überweisung, zahlungsoptionen, bank, einmalig",4,2025-08-30 04:48:00.000


### Apollo — Dynamic Topic Modelling (over time)


In [ ]:
# Topics over time (same as RT_de)
apollo_topics_over_time = apollo_topic_model.topics_over_time(
    docs=apollo_docs,
    timestamps=apollo_timestamps,
    nr_bins=20,
)

display(apollo_topics_over_time.head(20))

display(apollo_topic_model.visualize_topics_over_time(
    apollo_topics_over_time,
    top_n_topics=10,
))


## Compact — Topic Modelling (same pipeline as RT_de)


In [128]:
# Compact topic modelling (same pipeline as RT_de)
# Uses the same cleaning, chunking, BERTopic setup, and outputs as RT_de.
if 'BERTopic' not in globals() or 'clean_topic_text' not in globals() or 'chunk_text' not in globals():
    raise RuntimeError('Please run the RT_de topic modelling cell first.')
compact_source_key = next((k for k in dfs_by_source if k.lower() == 'compact'), None)
if compact_source_key is None:
    raise KeyError(f"Source not found for Compact. Available sources: {list(dfs_by_source.keys())}")
compact_topic_df = dfs_by_source[compact_source_key].copy()
required_cols = {'Inhalt', 'Date'}
missing_cols = required_cols.difference(compact_topic_df.columns)
if missing_cols:
    raise KeyError(f"Missing required columns for Compact: {sorted(missing_cols)}")
# Keep only rows with available full text
compact_topic_df = compact_topic_df[compact_topic_df['Inhalt'].notna()].copy()
# Same topic columns as RT_de
compact_topic_df['topic_text_raw'] = compact_topic_df['Inhalt'].astype(str)
compact_topic_df['topic_date'] = pd.to_datetime(compact_topic_df['Date'], errors='coerce')
# Same cleaning pipeline as RT_de
compact_topic_df['topic_text_clean'] = compact_topic_df['topic_text_raw'].apply(clean_topic_text)
compact_topic_df['topic_text_model'] = compact_topic_df['topic_text_clean']
# Same chunking pipeline as RT_de
compact_topic_df['topic_chunks'] = compact_topic_df['topic_text_model'].apply(chunk_text)
compact_model_df = (
    compact_topic_df[['topic_date', 'topic_chunks']]
    .explode('topic_chunks')
    .rename(columns={'topic_chunks': 'topic_text_model'})
)
compact_model_df = compact_model_df[
    compact_model_df['topic_date'].notna()
    & compact_model_df['topic_text_model'].notna()
    & compact_model_df['topic_text_model'].str.len().gt(0)
].copy()
# Full corpus by default (no sampling)
MAX_DOCS = None
if MAX_DOCS is not None and len(compact_model_df) > MAX_DOCS:
    compact_model_df = compact_model_df.sample(MAX_DOCS, random_state=42)
compact_model_df = compact_model_df.sort_values('topic_date').reset_index(drop=True)
if compact_model_df.empty:
    raise ValueError('No usable chunks after preprocessing.')
compact_docs = compact_model_df['topic_text_model'].tolist()
compact_timestamps = compact_model_df['topic_date'].dt.to_pydatetime().tolist()
print(f'Compact articles used: {compact_topic_df["topic_date"].notna().sum()}')
print(f'Compact chunks used for BERTopic: {len(compact_docs)}')
print(f'Date range: {compact_model_df["topic_date"].min().date()} to {compact_model_df["topic_date"].max().date()}')
# Same sanity check as RT_de
compact_token_counts = Counter(' '.join(compact_topic_df['topic_text_clean']).split())
compact_top_tokens_df = pd.DataFrame(compact_token_counts.most_common(20), columns=['token', 'count'])
# Overall non-time topic overview (all documents/chunks)
display(compact_top_tokens_df)
# Same BERTopic configuration as RT_de
compact_topic_model = BERTopic(
    language='multilingual',
    embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    min_topic_size=25,
    calculate_probabilities=False,
    verbose=True,
)
compact_topics, _ = compact_topic_model.fit_transform(compact_docs)
compact_topic_info = compact_topic_model.get_topic_info()
display(compact_topic_info.head(20))
display(compact_topic_model.visualize_barchart(top_n_topics=12))



Compact articles used: 2045
Compact chunks used for BERTopic: 3768
Date range: 2025-06-11 to 2026-02-10


,token,count
0,compact,3399
1,afd,2088
2,deutschland,1358
3,us,1261
4,erfahren,1248
5,bestellen,1205
6,trump,1166
7,prozent,1131
8,q10,1013
9,foto,999


2026-02-19 20:00:47,306 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/118 [00:00<?, ?it/s]

2026-02-19 20:01:16,032 - BERTopic - Embedding - Completed ✓
2026-02-19 20:01:16,033 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 20:01:23,822 - BERTopic - Dimensionality - Completed ✓
2026-02-19 20:01:23,823 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 20:01:23,899 - BERTopic - Cluster - Completed ✓
2026-02-19 20:01:23,901 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 20:01:24,249 - BERTopic - Representation - Completed ✓


Topic  Count                                     Name  \
0      -1   1122            -1_afd_compact_deutschland_us   
1       0    533       0_geschichte_deutschen_compact_afd   
2       1    418    1_q10_entzündungen_stress_astaxanthin   
3       2    280   2_passwort_printausgabe_desktop_loggen   
4       3    273           3_russland_ukraine_putin_krieg   
5       4    166  4_israel_netanjahu_israelischen_israels   
6       5    117       5_compact_tv_unterstuetzen_spenden   
7       6    108              6_asyl_euro_deutschland_afd   
8       7     90     7_silber_gold_dollar_silbermedaillen   
9       8     90          8_antifa_hammerbande_maja_szene   
10      9     88             9_messer_morde_opfer_polizei   
11     10     81                    10_afd_cdu_spd_partei   
12     11     78               11_venezuela_maduro_us_usa   
13     12     47         12_corona_witzschel_impfung_piks   
14     13     45              13_euro_brd_diktatur_seiten   
15     14     40          14_polizei_täter_messer_mädchen   
16     15     38          15_compact_bestellen_afd_verbot   
17     16     37                16_prozent_afd_bsw_partei   
18     17     34         17_epstein_trump_epsteins_greene   
19     18     29                18_mars_ufo_planeten_musk   

                                                                                                       Representation  \
0                                     [afd, compact, deutschland, us, trump, bestellen, foto, erfahren, prozent, usa]   
1                       [geschichte, deutschen, compact, afd, foto, deutsche, deutschland, erfahren, bestellen, buch]   
2                [q10, entzündungen, stress, astaxanthin, vitamin, körper, zellen, immunsystem, magnesium, ernährung]   
3   [passwort, printausgabe, desktop, loggen, benutzern, zurückzusetzen, benutzername, klicke, tablet, registrierten]   
4                                  [russland, ukraine, putin, krieg, trump, nato, russischen, frieden, russische, eu]   
5                               [israel, netanjahu, israelischen, israels, gaza, iran, trump, israelische, hamas, us]   
6                               [compact, tv, unterstuetzen, spenden, gratis, unterstützung, bitte, klemm, paul, afd]   
7                         [asyl, euro, deutschland, afd, migranten, invasion, paket, merkel, asylbewerber, bestellen]   
8                               [silber, gold, dollar, silbermedaillen, euro, prozent, compact, preis, us, medaillen]   
9               [antifa, hammerbande, maja, szene, linksextremisten, linksextreme, compact, anhänger, dresden, linke]   
10                                 [messer, morde, opfer, polizei, mädchen, täter, gewalt, spezial, compact, jährige]   
11                                              [afd, cdu, spd, partei, prozent, weidel, alice, merz, union, stimmen]   
12                         [venezuela, maduro, us, usa, präsident, venezuelas, washington, trump, caracas, rodriguez]   
13                             [corona, witzschel, impfung, piks, dr, aufarbeitung, bianca, dvd, maßnahmen, pandemie]   
14                                          [euro, brd, diktatur, seiten, afd, spd, paket, demokratie, merz, brosius]   
15                           [polizei, täter, messer, mädchen, jährige, opfer, angreifer, konarek, mann, eigenschutz]   
16                       [compact, bestellen, afd, verbot, demokratie, brosius, meinungsfreiheit, corona, brd, spahn]   
17                                     [prozent, afd, bsw, partei, wähler, wagenknecht, politik, sachsen, kickl, fpö]   
18                                    [epstein, trump, epsteins, greene, akten, jeffrey, maxwell, us, trumps, mossad]   
19                            [mars, ufo, planeten, musk, außerirdischen, spacex, erde, däniken, antarktis, starship]   

                                                                                                                                                                                    

20it [00:04,  4.47it/s]


,Topic,Words,Frequency,Timestamp
0,-1,"soros, compact, us, welt, prozent",67,2025-06-10 18:08:38.400
1,0,"inka, geschichte, dionysos, normannen, foto",32,2025-06-10 18:08:38.400
2,1,"q10, astaxanthin, stress, entzündungen, zucker",16,2025-06-10 18:08:38.400
3,2,"passwort, printausgabe, klicke, desktop, benutzername",10,2025-06-10 18:08:38.400
4,3,"russland, frieden, spd, putin, trump",11,2025-06-10 18:08:38.400
5,4,"israel, iran, israelischen, netanjahu, israels",42,2025-06-10 18:08:38.400
6,5,"compact, unterstuetzen, spenden, tv, gratis",4,2025-06-10 18:08:38.400
7,6,"euro, asyl, kirchenasyl, abschiebungen, asylbewerber",8,2025-06-10 18:08:38.400
8,7,"blockchain, technologie, löbe, finanzämtern, zuschauergalerie",1,2025-06-10 18:08:38.400
9,8,"maja, antifa, hammerbande, ungarn, behörden",7,2025-06-10 18:08:38.400


### Compact — Dynamic Topic Modelling (over time)


In [ ]:
# Topics over time (same as RT_de)
compact_topics_over_time = compact_topic_model.topics_over_time(
    docs=compact_docs,
    timestamps=compact_timestamps,
    nr_bins=20,
)

display(compact_topics_over_time.head(20))

display(compact_topic_model.visualize_topics_over_time(
    compact_topics_over_time,
    top_n_topics=10,
))


## deusch_pravda — Topic Modelling (same pipeline as RT_de)


In [129]:
# deusch_pravda topic modelling (same pipeline as RT_de)
# Uses the same cleaning, chunking, BERTopic setup, and outputs as RT_de.
if 'BERTopic' not in globals() or 'clean_topic_text' not in globals() or 'chunk_text' not in globals():
    raise RuntimeError('Please run the RT_de topic modelling cell first.')
deusch_pravda_source_key = next((k for k in dfs_by_source if k.lower() == 'deusch_pravda'), None)
if deusch_pravda_source_key is None:
    raise KeyError(f"Source not found for deusch_pravda. Available sources: {list(dfs_by_source.keys())}")
deusch_pravda_topic_df = dfs_by_source[deusch_pravda_source_key].copy()
required_cols = {'full_text', 'time'}
missing_cols = required_cols.difference(deusch_pravda_topic_df.columns)
if missing_cols:
    raise KeyError(f"Missing required columns for deusch_pravda: {sorted(missing_cols)}")
# Keep only rows with available full text
deusch_pravda_topic_df = deusch_pravda_topic_df[deusch_pravda_topic_df['full_text'].notna()].copy()
# Same topic columns as RT_de
deusch_pravda_topic_df['topic_text_raw'] = deusch_pravda_topic_df['full_text'].astype(str)
deusch_pravda_topic_df['topic_date'] = pd.to_datetime(deusch_pravda_topic_df['time'], errors='coerce')
# Same cleaning pipeline as RT_de
deusch_pravda_topic_df['topic_text_clean'] = deusch_pravda_topic_df['topic_text_raw'].apply(clean_topic_text)
deusch_pravda_topic_df['topic_text_model'] = deusch_pravda_topic_df['topic_text_clean']
# Same chunking pipeline as RT_de
deusch_pravda_topic_df['topic_chunks'] = deusch_pravda_topic_df['topic_text_model'].apply(chunk_text)
deusch_pravda_model_df = (
    deusch_pravda_topic_df[['topic_date', 'topic_chunks']]
    .explode('topic_chunks')
    .rename(columns={'topic_chunks': 'topic_text_model'})
)
deusch_pravda_model_df = deusch_pravda_model_df[
    deusch_pravda_model_df['topic_date'].notna()
    & deusch_pravda_model_df['topic_text_model'].notna()
    & deusch_pravda_model_df['topic_text_model'].str.len().gt(0)
].copy()
# Full corpus by default (no sampling)
MAX_DOCS = None
if MAX_DOCS is not None and len(deusch_pravda_model_df) > MAX_DOCS:
    deusch_pravda_model_df = deusch_pravda_model_df.sample(MAX_DOCS, random_state=42)
deusch_pravda_model_df = deusch_pravda_model_df.sort_values('topic_date').reset_index(drop=True)
if deusch_pravda_model_df.empty:
    raise ValueError('No usable chunks after preprocessing.')
deusch_pravda_docs = deusch_pravda_model_df['topic_text_model'].tolist()
deusch_pravda_timestamps = deusch_pravda_model_df['topic_date'].dt.to_pydatetime().tolist()
print(f'deusch_pravda articles used: {deusch_pravda_topic_df["topic_date"].notna().sum()}')
print(f'deusch_pravda chunks used for BERTopic: {len(deusch_pravda_docs)}')
print(f'Date range: {deusch_pravda_model_df["topic_date"].min().date()} to {deusch_pravda_model_df["topic_date"].max().date()}')
# Same sanity check as RT_de
deusch_pravda_token_counts = Counter(' '.join(deusch_pravda_topic_df['topic_text_clean']).split())
deusch_pravda_top_tokens_df = pd.DataFrame(deusch_pravda_token_counts.most_common(20), columns=['token', 'count'])
# Overall non-time topic overview (all documents/chunks)
display(deusch_pravda_top_tokens_df)
# Same BERTopic configuration as RT_de
deusch_pravda_topic_model = BERTopic(
    language='multilingual',
    embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    min_topic_size=25,
    calculate_probabilities=False,
    verbose=True,
)
deusch_pravda_topics, _ = deusch_pravda_topic_model.fit_transform(deusch_pravda_docs)
deusch_pravda_topic_info = deusch_pravda_topic_model.get_topic_info()
display(deusch_pravda_topic_info.head(20))
display(deusch_pravda_topic_model.visualize_barchart(top_n_topics=12))



deusch_pravda articles used: 32850
deusch_pravda chunks used for BERTopic: 35431
Date range: 2025-01-06 to 2025-12-07


,token,count
0,ukraine,178653
1,russland,119802
2,russischen,77525
3,usa,74976
4,trump,74364
5,us,67358
6,deutschland,58235
7,eu,57593
8,russische,55784
9,ukrainischen,53919


2026-02-19 20:02:02,554 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1108 [00:00<?, ?it/s]

2026-02-19 20:05:20,252 - BERTopic - Embedding - Completed ✓
2026-02-19 20:05:20,254 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 20:05:26,398 - BERTopic - Dimensionality - Completed ✓
2026-02-19 20:05:26,401 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 20:05:29,236 - BERTopic - Cluster - Completed ✓
2026-02-19 20:05:29,245 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 20:05:30,799 - BERTopic - Representation - Completed ✓


Topic  Count                                           Name  \
0      -1  18158    -1_ukraine_russland_russischen_ukrainischen   
1       0   1211                0_israel_iran_israelischen_gaza   
2       1    977               1_patriot_ukraine_waffen_raketen   
3       2    922                          2_afd_merz_partei_cdu   
4       3    728         3_euro_prozent_deutschland_unternehmen   
5       4    628            4_drohnen_region_nacht_streitkräfte   
6       5    589    5_migranten_migration_grenzkontrollen_polen   
7       6    500         6_deutschen_familie_geschichte_geboren   
8       7    432   7_streitkräfte_truppen_richtung_ukrainischen   
9       8    371  8_sanktionen_sanktionspaket_russland_slowakei   
10      9    360        9_verhandlungen_putin_istanbul_wladimir   
11     10    359                     10_musk_elon_epstein_trump   
12     11    312  11_deutschland_deutschen_wolodin_deutschlands   
13     12    252             12_covid_mrna_impfstoffe_impfstoff   
14     13    236    13_sieges_vaterländischen_fand_sowjetischen   
15     14    225             14_rakete_flugzeug_ausgestattet_su   
16     15    222         15_angeles_proteste_newsom_kalifornien   
17     16    220           16_ki_digitale_intelligenz_digitalen   
18     17    216                17_china_zölle_chinesische_genf   
19     18    206           18_gas_lng_gaslieferungen_russisches   

                                                                                                           Representation  \
0                         [ukraine, russland, russischen, ukrainischen, deutschland, russische, trump, usa, putin, krieg]   
1                            [israel, iran, israelischen, gaza, israelische, gazastreifen, israels, netanjahu, hamas, al]   
2                   [patriot, ukraine, waffen, raketen, lieferungen, lieferung, waffenlieferungen, taurus, kiew, systeme]   
3                                     [afd, merz, partei, cdu, spd, verfassungsschutz, stimmen, adh, wahlgang, bundestag]   
4         [euro, prozent, deutschland, unternehmen, industrie, milliarden, wirtschaft, achtsamkeit, erich, arbeitsplätze]   
5               [drohnen, region, nacht, streitkräfte, angriff, angriffe, ukrainischen, explosionen, gebiet, ukrainische]   
6   [migranten, migration, grenzkontrollen, polen, deutschland, grenze, grenzen, dobrindt, kontrollen, migrationspolitik]   
7                   [deutschen, familie, geschichte, geboren, deutsche, vaterländischen, krim, deutschland, stadt, krieg]   
8                     [streitkräfte, truppen, richtung, ukrainischen, feind, gebiet, russischen, region, sumy, russische]   
9             [sanktionen, sanktionspaket, russland, slowakei, eu, waffenruhe, paket, europäische, kommission, verhängen]   
10                    [verhandlungen, putin, istanbul, wladimir, ukraine, waffenstillstand, putins, peskow, kiew, macron]   
11                                             [musk, elon, epstein, trump, tesla, musks, milliardär, spacex, partei, us]   
12     [deutschland, deutschen, wolodin, deutschlands, nationalsozialismus, staatsduma, weltkriegs, nazi, russland, merz]   
13                                 [covid, mrna, impfstoffe, impfstoff, studie, impfung, jr, kennedy, virus, impfstoffen]   
14         [sieges, vaterländischen, fand, sowjetischen, veteranen, denkmal, soldaten, veranstaltung, nahmen, landsleute]   
15                        [rakete, flugzeug, ausgestattet, su, flugzeuge, tests, entwickelt, entwicklung, piloten, start]   
16              [angeles, proteste, newsom, kalifornien, nationalgarde, unruhen, gouverneur, trump, demonstranten, gavin]   
17                                   [ki, digitale, intelligenz, digitalen, daten, krypto, künstliche, eu, dsa, palantir]   
18                                          [china, zölle, chinesische, genf, xi, peking, chinesischen, usa, bessent, he]   
19                          [gas, lng, gaslieferungen, russisches, kubikmeter, eu, russischem, e

14it [00:26,  1.87s/it]


,Topic,Words,Frequency,Timestamp
0,-1,"ukraine, russland, russischen, ukrainischen, russische",1650,2025-01-05 15:59:09.840
1,0,"israel, gaza, iran, gazastreifen, israels",113,2025-01-05 15:59:09.840
2,1,"patriot, taurus, ukraine, waffen, raketen",76,2025-01-05 15:59:09.840
3,2,"afd, partei, kandidaten, cdu, spd",38,2025-01-05 15:59:09.840
4,3,"euro, deutschland, milliarden, achtsamkeit, erich",75,2025-01-05 15:59:09.840
5,4,"drohnen, region, angriff, nacht, streitkräfte",71,2025-01-05 15:59:09.840
6,5,"migranten, polen, grenzkontrollen, tusk, migration",47,2025-01-05 15:59:09.840
7,6,"kirche, august, lilie, starb, denisovna",48,2025-01-05 15:59:09.840
8,7,"streitkräfte, truppen, richtung, ukrainischen, sumy",47,2025-01-05 15:59:09.840
9,8,"sanktionen, sanktionspaket, russland, paket, öl",18,2025-01-05 15:59:09.840


### deusch_pravda — Dynamic Topic Modelling (over time)


In [ ]:
# Topics over time (same as RT_de)
deusch_pravda_topics_over_time = deusch_pravda_topic_model.topics_over_time(
    docs=deusch_pravda_docs,
    timestamps=deusch_pravda_timestamps,
    nr_bins=20,
)

display(deusch_pravda_topics_over_time.head(20))

display(deusch_pravda_topic_model.visualize_topics_over_time(
    deusch_pravda_topics_over_time,
    top_n_topics=10,
))


## Nius Rohdaten — Topic Modelling (same pipeline as RT_de)


In [130]:
# Nius Rohdaten topic modelling (same pipeline as RT_de)
# Uses the same cleaning, chunking, BERTopic setup, and outputs as RT_de.
if 'BERTopic' not in globals() or 'clean_topic_text' not in globals() or 'chunk_text' not in globals():
    raise RuntimeError('Please run the RT_de topic modelling cell first.')
nius_rohdaten_source_key = next((k for k in dfs_by_source if k.lower() == 'nius_rohdaten_neu'), None)
if nius_rohdaten_source_key is None:
    raise KeyError(f"Source not found for Nius Rohdaten. Available sources: {list(dfs_by_source.keys())}")
nius_rohdaten_topic_df = dfs_by_source[nius_rohdaten_source_key].copy()
required_cols = {'article_text', 'day'}
missing_cols = required_cols.difference(nius_rohdaten_topic_df.columns)
if missing_cols:
    raise KeyError(f"Missing required columns for Nius Rohdaten: {sorted(missing_cols)}")
# Keep only rows with available full text
nius_rohdaten_topic_df = nius_rohdaten_topic_df[nius_rohdaten_topic_df['article_text'].notna()].copy()
# Same topic columns as RT_de
nius_rohdaten_topic_df['topic_text_raw'] = nius_rohdaten_topic_df['article_text'].astype(str)
nius_rohdaten_topic_df['topic_date'] = pd.to_datetime(nius_rohdaten_topic_df['day'], errors='coerce')
# Same cleaning pipeline as RT_de
nius_rohdaten_topic_df['topic_text_clean'] = nius_rohdaten_topic_df['topic_text_raw'].apply(clean_topic_text)
nius_rohdaten_topic_df['topic_text_model'] = nius_rohdaten_topic_df['topic_text_clean']
# Same chunking pipeline as RT_de
nius_rohdaten_topic_df['topic_chunks'] = nius_rohdaten_topic_df['topic_text_model'].apply(chunk_text)
nius_rohdaten_model_df = (
    nius_rohdaten_topic_df[['topic_date', 'topic_chunks']]
    .explode('topic_chunks')
    .rename(columns={'topic_chunks': 'topic_text_model'})
)
nius_rohdaten_model_df = nius_rohdaten_model_df[
    nius_rohdaten_model_df['topic_date'].notna()
    & nius_rohdaten_model_df['topic_text_model'].notna()
    & nius_rohdaten_model_df['topic_text_model'].str.len().gt(0)
].copy()
# Full corpus by default (no sampling)
MAX_DOCS = None
if MAX_DOCS is not None and len(nius_rohdaten_model_df) > MAX_DOCS:
    nius_rohdaten_model_df = nius_rohdaten_model_df.sample(MAX_DOCS, random_state=42)
nius_rohdaten_model_df = nius_rohdaten_model_df.sort_values('topic_date').reset_index(drop=True)
if nius_rohdaten_model_df.empty:
    raise ValueError('No usable chunks after preprocessing.')
nius_rohdaten_docs = nius_rohdaten_model_df['topic_text_model'].tolist()
nius_rohdaten_timestamps = nius_rohdaten_model_df['topic_date'].dt.to_pydatetime().tolist()
print(f'Nius Rohdaten articles used: {nius_rohdaten_topic_df["topic_date"].notna().sum()}')
print(f'Nius Rohdaten chunks used for BERTopic: {len(nius_rohdaten_docs)}')
print(f'Date range: {nius_rohdaten_model_df["topic_date"].min().date()} to {nius_rohdaten_model_df["topic_date"].max().date()}')
# Same sanity check as RT_de
nius_rohdaten_token_counts = Counter(' '.join(nius_rohdaten_topic_df['topic_text_clean']).split())
nius_rohdaten_top_tokens_df = pd.DataFrame(nius_rohdaten_token_counts.most_common(20), columns=['token', 'count'])
# Overall non-time topic overview (all documents/chunks)
display(nius_rohdaten_top_tokens_df)
# Same BERTopic configuration as RT_de
nius_rohdaten_topic_model = BERTopic(
    language='multilingual',
    embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    min_topic_size=25,
    calculate_probabilities=False,
    verbose=True,
)
nius_rohdaten_topics, _ = nius_rohdaten_topic_model.fit_transform(nius_rohdaten_docs)
nius_rohdaten_topic_info = nius_rohdaten_topic_model.get_topic_info()
display(nius_rohdaten_topic_info.head(20))
display(nius_rohdaten_topic_model.visualize_barchart(top_n_topics=12))



Nius Rohdaten articles used: 4872
Nius Rohdaten chunks used for BERTopic: 9377
Date range: 2025-05-01 to 2026-02-10


,token,count
0,nius,7150
1,deutschland,6351
2,merz,5260
3,prozent,4754
4,euro,4533
5,afd,4522
6,menschen,4008
7,cdu,3947
8,spd,3790
9,deutschen,3184


2026-02-19 20:06:01,898 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/294 [00:00<?, ?it/s]

2026-02-19 20:07:06,661 - BERTopic - Embedding - Completed ✓
2026-02-19 20:07:06,662 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 20:07:09,253 - BERTopic - Dimensionality - Completed ✓
2026-02-19 20:07:09,254 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 20:07:09,525 - BERTopic - Cluster - Completed ✓
2026-02-19 20:07:09,529 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 20:07:10,978 - BERTopic - Representation - Completed ✓


Topic  Count                             Name  \
0      0   9122  0_deutschland_nius_merz_prozent   
1      1    255     1_nius_live_moderator_studio   

                                                                             Representation  \
0              [deutschland, nius, merz, prozent, euro, afd, menschen, cdu, spd, deutschen]   
1  [nius, live, moderator, studio, aufzeichnung, direkt, radio, sendung, themen, purrucker]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

20it [00:15,  1.33it/s]


,Topic,Words,Frequency,Timestamp
0,0,"afd, merz, deutschland, nius, verfassungsschutz",454,2025-04-30 17:09:36
1,1,"nius, live, studio, purrucker, alex",10,2025-04-30 17:09:36
2,0,"deutschland, merz, nius, menschen, afd",368,2025-05-15 06:00:00
3,1,"nius, live, moderator, studio, sendung",10,2025-05-15 06:00:00
4,0,"nius, deutschland, merz, prozent, trump",362,2025-05-29 12:00:00
5,1,"nius, live, studio, moderator, themen",4,2025-05-29 12:00:00
6,0,"deutschland, euro, nius, merz, israel",436,2025-06-12 18:00:00
7,1,"nius, live, sendung, themen, studio",10,2025-06-12 18:00:00
8,0,"merz, deutschland, euro, nius, cdu",406,2025-06-27 00:00:00
9,1,"nius, live, studio, purrucker, moderator",11,2025-06-27 00:00:00


### Nius Rohdaten — Dynamic Topic Modelling (over time)


In [ ]:
# Topics over time (same as RT_de)
nius_rohdaten_topics_over_time = nius_rohdaten_topic_model.topics_over_time(
    docs=nius_rohdaten_docs,
    timestamps=nius_rohdaten_timestamps,
    nr_bins=20,
)

display(nius_rohdaten_topics_over_time.head(20))

display(nius_rohdaten_topic_model.visualize_topics_over_time(
    nius_rohdaten_topics_over_time,
    top_n_topics=10,
))


## Tichy's Einblick — Topic Modelling (same pipeline as RT_de)


In [131]:
# Tichy's Einblick topic modelling (same pipeline as RT_de)
# Uses the same cleaning, chunking, BERTopic setup, and outputs as RT_de.
if 'BERTopic' not in globals() or 'clean_topic_text' not in globals() or 'chunk_text' not in globals():
    raise RuntimeError('Please run the RT_de topic modelling cell first.')
tichys_einblick_source_key = next((k for k in dfs_by_source if k.lower() == "tichy's einblick"), None)
if tichys_einblick_source_key is None:
    raise KeyError(f"Source not found for Tichy's Einblick. Available sources: {list(dfs_by_source.keys())}")
tichys_einblick_topic_df = dfs_by_source[tichys_einblick_source_key].copy()
required_cols = {'article_text', 'date'}
missing_cols = required_cols.difference(tichys_einblick_topic_df.columns)
if missing_cols:
    raise KeyError(f"Missing required columns for Tichy's Einblick: {sorted(missing_cols)}")
# Keep only rows with available full text
tichys_einblick_topic_df = tichys_einblick_topic_df[tichys_einblick_topic_df['article_text'].notna()].copy()
# Same topic columns as RT_de
tichys_einblick_topic_df['topic_text_raw'] = tichys_einblick_topic_df['article_text'].astype(str)
tichys_einblick_topic_df['topic_date'] = pd.to_datetime(tichys_einblick_topic_df['date'], errors='coerce')
# Same cleaning pipeline as RT_de
tichys_einblick_topic_df['topic_text_clean'] = tichys_einblick_topic_df['topic_text_raw'].apply(clean_topic_text)
tichys_einblick_topic_df['topic_text_model'] = tichys_einblick_topic_df['topic_text_clean']
# Same chunking pipeline as RT_de
tichys_einblick_topic_df['topic_chunks'] = tichys_einblick_topic_df['topic_text_model'].apply(chunk_text)
tichys_einblick_model_df = (
    tichys_einblick_topic_df[['topic_date', 'topic_chunks']]
    .explode('topic_chunks')
    .rename(columns={'topic_chunks': 'topic_text_model'})
)
tichys_einblick_model_df = tichys_einblick_model_df[
    tichys_einblick_model_df['topic_date'].notna()
    & tichys_einblick_model_df['topic_text_model'].notna()
    & tichys_einblick_model_df['topic_text_model'].str.len().gt(0)
].copy()
# Full corpus by default (no sampling)
MAX_DOCS = None
if MAX_DOCS is not None and len(tichys_einblick_model_df) > MAX_DOCS:
    tichys_einblick_model_df = tichys_einblick_model_df.sample(MAX_DOCS, random_state=42)
tichys_einblick_model_df = tichys_einblick_model_df.sort_values('topic_date').reset_index(drop=True)
if tichys_einblick_model_df.empty:
    raise ValueError('No usable chunks after preprocessing.')
tichys_einblick_docs = tichys_einblick_model_df['topic_text_model'].tolist()
tichys_einblick_timestamps = tichys_einblick_model_df['topic_date'].dt.to_pydatetime().tolist()
print(f"Tichy's Einblick articles used: {tichys_einblick_topic_df['topic_date'].notna().sum()}")
print(f"Tichy's Einblick chunks used for BERTopic: {len(tichys_einblick_docs)}")
print(f'Date range: {tichys_einblick_model_df["topic_date"].min().date()} to {tichys_einblick_model_df["topic_date"].max().date()}')
# Same sanity check as RT_de
tichys_einblick_token_counts = Counter(' '.join(tichys_einblick_topic_df['topic_text_clean']).split())
tichys_einblick_top_tokens_df = pd.DataFrame(tichys_einblick_token_counts.most_common(20), columns=['token', 'count'])
# Overall non-time topic overview (all documents/chunks)
display(tichys_einblick_top_tokens_df)
# Same BERTopic configuration as RT_de
tichys_einblick_topic_model = BERTopic(
    language='multilingual',
    embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    min_topic_size=25,
    calculate_probabilities=False,
    verbose=True,
)
tichys_einblick_topics, _ = tichys_einblick_topic_model.fit_transform(tichys_einblick_docs)
tichys_einblick_topic_info = tichys_einblick_topic_model.get_topic_info()
display(tichys_einblick_topic_info.head(20))
display(tichys_einblick_topic_model.visualize_barchart(top_n_topics=12))



Tichy's Einblick articles used: 3126
Tichy's Einblick chunks used for BERTopic: 7936
Date range: 2025-08-01 to 2026-02-11


,token,count
0,deutschland,5532
1,prozent,4900
2,merz,4182
3,eu,4167
4,euro,3777
5,spd,3340
6,deutschen,2918
7,cdu,2886
8,politik,2783
9,afd,2666


2026-02-19 20:07:27,980 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/248 [00:00<?, ?it/s]

2026-02-19 20:09:18,123 - BERTopic - Embedding - Completed ✓
2026-02-19 20:09:18,124 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 20:09:20,427 - BERTopic - Dimensionality - Completed ✓
2026-02-19 20:09:20,428 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 20:09:20,629 - BERTopic - Cluster - Completed ✓
2026-02-19 20:09:20,635 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 20:09:21,481 - BERTopic - Representation - Completed ✓


Topic  Count                                    Name  \
0     -1     13   -1_mattheis_korrespondent_tichy_asien   
1      0   7752           0_deutschland_prozent_eu_merz   
2      1    171  1_soundcloud_spotify_abonnieren_amazon   

                                                                                              Representation  \
0  [mattheis, korrespondent, tichy, asien, roland, news, tichys, wirtschaftswoche, wirtschaft, börsenwecker]   
1                                  [deutschland, prozent, eu, merz, euro, spd, deutschen, cdu, politik, afd]   
2        [soundcloud, spotify, abonnieren, amazon, itunes, einverstanden, angezeigt, te, inhalte, gutschein]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

20it [00:17,  1.17it/s]


,Topic,Words,Frequency,Timestamp
0,0,"deutschland, prozent, euro, spd, merz",366,2025-07-31 19:20:38.400
1,1,"spotify, itunes, soundcloud, abonnieren, amazon",9,2025-07-31 19:20:38.400
2,0,"deutschland, prozent, euro, eu, merz",344,2025-08-10 16:48:00.000
3,1,"spotify, itunes, soundcloud, abonnieren, amazon",10,2025-08-10 16:48:00.000
4,0,"deutschland, prozent, euro, spd, eu",377,2025-08-20 09:36:00.000
5,1,"spotify, itunes, soundcloud, abonnieren, amazon",8,2025-08-20 09:36:00.000
6,0,"deutschland, prozent, euro, spd, merz",342,2025-08-30 02:24:00.000
7,1,"spotify, itunes, soundcloud, abonnieren, amazon",8,2025-08-30 02:24:00.000
8,0,"deutschland, prozent, kirk, spd, euro",416,2025-09-08 19:12:00.000
9,1,"spotify, itunes, soundcloud, abonnieren, amazon",9,2025-09-08 19:12:00.000


### Tichy's Einblick — Dynamic Topic Modelling (over time)


In [ ]:
# Topics over time (same as RT_de)
tichys_einblick_topics_over_time = tichys_einblick_topic_model.topics_over_time(
    docs=tichys_einblick_docs,
    timestamps=tichys_einblick_timestamps,
    nr_bins=20,
)

display(tichys_einblick_topics_over_time.head(20))

display(tichys_einblick_topic_model.visualize_topics_over_time(
    tichys_einblick_topics_over_time,
    top_n_topics=10,
))


## Reitschuster — Topic Modelling (same pipeline as RT_de)


In [132]:
# Reitschuster topic modelling (same pipeline as RT_de)
# Uses the same cleaning, chunking, BERTopic setup, and outputs as RT_de.
if 'BERTopic' not in globals() or 'clean_topic_text' not in globals() or 'chunk_text' not in globals():
    raise RuntimeError('Please run the RT_de topic modelling cell first.')
reitschuster_source_key = next((k for k in dfs_by_source if k.lower() == 'reitschuster'), None)
if reitschuster_source_key is None:
    raise KeyError(f"Source not found for Reitschuster. Available sources: {list(dfs_by_source.keys())}")
reitschuster_topic_df = dfs_by_source[reitschuster_source_key].copy()
required_cols = {'Inhalt', 'Date'}
missing_cols = required_cols.difference(reitschuster_topic_df.columns)
if missing_cols:
    raise KeyError(f"Missing required columns for Reitschuster: {sorted(missing_cols)}")
# Keep only rows with available full text
reitschuster_topic_df = reitschuster_topic_df[reitschuster_topic_df['Inhalt'].notna()].copy()
# Same topic columns as RT_de
reitschuster_topic_df['topic_text_raw'] = reitschuster_topic_df['Inhalt'].astype(str)
reitschuster_topic_df['topic_date'] = pd.to_datetime(reitschuster_topic_df['Date'], errors='coerce')
# Same cleaning pipeline as RT_de
reitschuster_topic_df['topic_text_clean'] = reitschuster_topic_df['topic_text_raw'].apply(clean_topic_text)
reitschuster_topic_df['topic_text_model'] = reitschuster_topic_df['topic_text_clean']
# Same chunking pipeline as RT_de
reitschuster_topic_df['topic_chunks'] = reitschuster_topic_df['topic_text_model'].apply(chunk_text)
reitschuster_model_df = (
    reitschuster_topic_df[['topic_date', 'topic_chunks']]
    .explode('topic_chunks')
    .rename(columns={'topic_chunks': 'topic_text_model'})
)
reitschuster_model_df = reitschuster_model_df[
    reitschuster_model_df['topic_date'].notna()
    & reitschuster_model_df['topic_text_model'].notna()
    & reitschuster_model_df['topic_text_model'].str.len().gt(0)
].copy()
# Full corpus by default (no sampling)
MAX_DOCS = None
if MAX_DOCS is not None and len(reitschuster_model_df) > MAX_DOCS:
    reitschuster_model_df = reitschuster_model_df.sample(MAX_DOCS, random_state=42)
reitschuster_model_df = reitschuster_model_df.sort_values('topic_date').reset_index(drop=True)
if reitschuster_model_df.empty:
    raise ValueError('No usable chunks after preprocessing.')
reitschuster_docs = reitschuster_model_df['topic_text_model'].tolist()
reitschuster_timestamps = reitschuster_model_df['topic_date'].dt.to_pydatetime().tolist()
print(f'Reitschuster articles used: {reitschuster_topic_df["topic_date"].notna().sum()}')
print(f'Reitschuster chunks used for BERTopic: {len(reitschuster_docs)}')
print(f'Date range: {reitschuster_model_df["topic_date"].min().date()} to {reitschuster_model_df["topic_date"].max().date()}')
# Same sanity check as RT_de
reitschuster_token_counts = Counter(' '.join(reitschuster_topic_df['topic_text_clean']).split())
reitschuster_top_tokens_df = pd.DataFrame(reitschuster_token_counts.most_common(20), columns=['token', 'count'])
# Overall non-time topic overview (all documents/chunks)
display(reitschuster_top_tokens_df)
# Same BERTopic configuration as RT_de
reitschuster_topic_model = BERTopic(
    language='multilingual',
    embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    min_topic_size=25,
    calculate_probabilities=False,
    verbose=True,
)
reitschuster_topics, _ = reitschuster_topic_model.fit_transform(reitschuster_docs)
reitschuster_topic_info = reitschuster_topic_model.get_topic_info()
display(reitschuster_topic_info.head(20))
display(reitschuster_topic_model.visualize_barchart(top_n_topics=12))



Reitschuster articles used: 520
Reitschuster chunks used for BERTopic: 2424
Date range: 2025-06-10 to 2026-02-10


,token,count
0,rumble,7748
1,function,5244
2,push,2623
3,document,2623
4,bitte,2069
5,video,2045
6,afd,1966
7,arguments,1951
8,play,1938
9,if,1937


2026-02-19 20:09:39,483 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/76 [00:00<?, ?it/s]

2026-02-19 20:10:01,099 - BERTopic - Embedding - Completed ✓
2026-02-19 20:10:01,100 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 20:10:04,525 - BERTopic - Dimensionality - Completed ✓
2026-02-19 20:10:04,526 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 20:10:04,581 - BERTopic - Cluster - Completed ✓
2026-02-19 20:10:04,584 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 20:10:04,836 - BERTopic - Representation - Completed ✓


Topic  Count                                  Name  \
0     -1     64       -1_rumble_function_div_document   
1      0   1844   0_function_seite_rumble_deutschland   
2      1    167        1_rumble_function_div_document   
3      2     85    2_rumble_function_arguments_length   
4      3     74    3_rumble_function_arguments_script   
5      4     61    4_rumble_function_v6wfa04_kritiker   
6      5     56  5_rumble_function_v6vlm0e_entmündigt   
7      6     44            6_kelle_rebmann_kai_regeln   
8      7     29     7_rumble_augsburg_function_terror   

                                                                                                     Representation  \
0                            [rumble, function, div, document, video, play, secondsloading, src, seconds15, script]   
1                             [function, seite, rumble, deutschland, zeigt, afd, menschen, kritisch, bleibt, trägt]   
2                           [rumble, function, div, document, video, play, secondsloading, seconds15, push, script]   
3  [rumble, function, arguments, length, if, getelementsbytagname, async, createelement, secondsloading, seconds15]   
4                  [rumble, function, arguments, script, div, length, if, src, getelementsbytagname, createelement]   
5                          [rumble, function, v6wfa04, kritiker, div, video, play, lobt, secondsloading, seconds15]   
6                      [rumble, function, v6vlm0e, entmündigt, demokratie, sieg, sichert, kandidat, gericht, farce]   
7                    [kelle, rebmann, kai, regeln, kommentar, kommentarfunktion, meinung, klaus, rumble, lengsfeld]   
8                  [rumble, augsburg, function, terror, unheimlich, daheim, weihnachten, v718wgc, v71dv2m, pollern]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

18it [00:02,  7.62it/s]


,Topic,Words,Frequency,Timestamp
0,-1,"rumble, function, div, play, video",11,2025-06-09 18:07:12
1,0,"seite, function, rumble, menschen, medien",132,2025-06-09 18:07:12
2,1,"rumble, function, div, document, play",19,2025-06-09 18:07:12
3,2,"rumble, function, if, length, createelement",5,2025-06-09 18:07:12
4,3,"rumble, function, arguments, script, length",6,2025-06-09 18:07:12
5,6,"rebmann, kai, meinung, regeln, kommentarfunktion",2,2025-06-09 18:07:12
6,-1,"rumble, function, div, document, play",10,2025-06-22 06:00:00
7,0,"function, seite, rumble, afd, menschen",117,2025-06-22 06:00:00
8,1,"rumble, function, document, div, play",16,2025-06-22 06:00:00
9,2,"rumble, function, if, length, createelement",5,2025-06-22 06:00:00


### Reitschuster — Dynamic Topic Modelling (over time)


In [ ]:
# Topics over time (same as RT_de)
reitschuster_topics_over_time = reitschuster_topic_model.topics_over_time(
    docs=reitschuster_docs,
    timestamps=reitschuster_timestamps,
    nr_bins=20,
)

display(reitschuster_topics_over_time.head(20))

display(reitschuster_topic_model.visualize_topics_over_time(
    reitschuster_topics_over_time,
    top_n_topics=10,
))


## Cross-Source Topic Comparison (Initial Exploration)


In [133]:
# Cross-Source Topic Comparison (Initial Exploration)

source_topic_infos = {
    'RT_de': topic_info,
    'Antispiegel': antispiegel_topic_info,
    'Apollo': apollo_topic_info,
    'Compact': compact_topic_info,
    'deusch_pravda': deusch_pravda_topic_info,
    'Nius Rohdaten': nius_rohdaten_topic_info,
    "Tichy's Einblick": tichys_einblick_topic_info,
    'Reitschuster': reitschuster_topic_info,
}

def top_topic_names(df, n=8):
    # Exclude outlier topic (-1) for comparison readability
    subset = df[df['Topic'] != -1].head(n)
    return subset['Name'].astype(str).tolist()

def topic_keywords(df, n=8):
    names = top_topic_names(df, n=n)
    kws = set()
    for name in names:
        for part in name.split('_'):
            part = part.strip().lower()
            if not part or part.isdigit() or len(part) < 2:
                continue
            kws.add(part)
    return kws

# Side-by-side top topic names
side_by_side = pd.DataFrame({
    source: pd.Series(top_topic_names(df, n=8))
    for source, df in source_topic_infos.items()
})
display(side_by_side)

# Overlapping vs unique keywords
source_keywords = {source: topic_keywords(df, n=8) for source, df in source_topic_infos.items()}
keyword_counts = Counter()
for kws in source_keywords.values():
    keyword_counts.update(kws)

overlap_rows = [
    {'keyword': kw, 'sources_count': cnt}
    for kw, cnt in keyword_counts.items()
    if cnt >= 2
]
overlap_df = pd.DataFrame(overlap_rows).sort_values('sources_count', ascending=False).head(30)
display(overlap_df)

unique_rows = []
for source, kws in source_keywords.items():
    uniques = sorted([kw for kw in kws if keyword_counts[kw] == 1])
    unique_rows.append({'source': source, 'unique_keywords': ', '.join(uniques[:12])})
unique_df = pd.DataFrame(unique_rows)
display(unique_df)

print('Overlapping themes: keywords appearing in multiple sources (see overlap_df).')
print('Unique themes: source-specific keywords (see unique_df).')
print("Framing differences: compare each source's top topic names side-by-side in side_by_side.")

,RT_de,Antispiegel,Apollo,Compact,deusch_pravda,Nius Rohdaten,Tichy's Einblick,Reitschuster
0,0_afd_spd_cdu_prozent,0_russland_ukraine_antworten_anmelden,0_werbung_pay_news_apollo,0_geschichte_deutschen_compact_afd,0_israel_iran_israelischen_gaza,0_deutschland_nius_merz_prozent,0_deutschland_prozent_eu_merz,0_function_seite_rumble_deutschland
1,1_putin_ukraine_trump_präsident,1_israel_gaza_israelische_hamas,1_pay_überweisung_zahlungsoptionen_bank,1_q10_entzündungen_stress_astaxanthin,1_patriot_ukraine_waffen_raketen,1_nius_live_moderator_studio,1_soundcloud_spotify_abonnieren_amazon,1_rumble_function_div_document
2,2_israel_hamas_gaza_gazastreifen,2_moldawien_wahlen_sandu_rumänien,2_pay_überweisung_zahlungsoptionen_bank,2_passwort_printausgabe_desktop_loggen,2_afd_merz_partei_cdu,NaN,NaN,2_rumble_function_arguments_length
3,3_venezuela_maduro_us_usa,3_venezuela_usa_trump_us,3_pay_überweisung_zahlungsoptionen_bank,3_russland_ukraine_putin_krieg,3_euro_prozent_deutschland_unternehmen,NaN,NaN,3_rumble_function_arguments_script
4,4_afd_deutschen_deutschland_deutsche,4_april_anmelden_antworten_menschen,4_pay_überweisung_zahlungsoptionen_bank,4_israel_netanjahu_israelischen_israels,4_drohnen_region_nacht_streitkräfte,NaN,NaN,4_rumble_function_v6wfa04_kritiker
5,5_selenskij_ukraine_selenskijs_ukrainischen,5_grönland_dänemark_usa_arktis,5_pay_überweisung_zahlungsoptionen_bank,5_compact_tv_unterstuetzen_spenden,5_migranten_migration_grenzkontrollen_polen,NaN,NaN,5_rumble_function_v6vlm0e_entmündigt
6,6_trump_musk_us_epstein,6_tacheles_teilen_sendung_freitag,NaN,6_asyl_euro_deutschland_afd,6_deutschen_familie_geschichte_geboren,NaN,NaN,6_kelle_rebmann_kai_regeln
7,7_prozent_euro_deutschland_deutsche,NaN,NaN,7_silber_gold_dollar_silbermedaillen,7_streitkräfte_truppen_richtung_ukrainischen,NaN,NaN,7_rumble_augsburg_function_terror


,keyword,sources_count
7,deutschland,6
2,prozent,4
14,ukraine,4
11,israel,4
0,afd,3
12,euro,3
8,gaza,3
19,merz,3
3,deutschen,3
6,usa,2


,source,unique_keywords
0,RT_de,"deutsche, epstein, gazastreifen, maduro, musk, präsident, selenskij, selenskijs, spd"
1,Antispiegel,"anmelden, antworten, april, arktis, dänemark, freitag, grönland, israelische, menschen, moldawien, rumänien, sandu"
2,Apollo,"apollo, bank, news, pay, werbung, zahlungsoptionen, überweisung"
3,Compact,"astaxanthin, asyl, compact, desktop, dollar, entzündungen, gold, israels, krieg, loggen, netanjahu, passwort"
4,deusch_pravda,"drohnen, familie, geboren, grenzkontrollen, iran, migranten, migration, nacht, partei, patriot, polen, raketen"
5,Nius Rohdaten,"live, moderator, nius, studio"
6,Tichy's Einblick,"abonnieren, amazon, eu, soundcloud, spotify"
7,Reitschuster,"arguments, augsburg, div, document, entmündigt, function, kai, kelle, kritiker, length, rebmann, regeln"


Overlapping themes: keywords appearing in multiple sources (see overlap_df).
Unique themes: source-specific keywords (see unique_df).
Framing differences: compare each source's top topic names side-by-side in side_by_side.


## Topic Bar Charts by Source (separate blocks)

Separate barchart cells (12 topics each) so each source chart renders independently.


### Antispiegel — Topic Word Scores (Top 12)


In [134]:
if 'antispiegel_topic_model' not in globals():
    raise RuntimeError('Run the corresponding source topic-modelling cell first.')
display(antispiegel_topic_model.visualize_barchart(top_n_topics=12))


### Apollo — Topic Word Scores (Top 12)


In [135]:
if 'apollo_topic_model' not in globals():
    raise RuntimeError('Run the corresponding source topic-modelling cell first.')
display(apollo_topic_model.visualize_barchart(top_n_topics=12))


### Compact — Topic Word Scores (Top 12)


In [136]:
if 'compact_topic_model' not in globals():
    raise RuntimeError('Run the corresponding source topic-modelling cell first.')
display(compact_topic_model.visualize_barchart(top_n_topics=12))


### deusch_pravda — Topic Word Scores (Top 12)


In [137]:
if 'deusch_pravda_topic_model' not in globals():
    raise RuntimeError('Run the corresponding source topic-modelling cell first.')
display(deusch_pravda_topic_model.visualize_barchart(top_n_topics=12))


### Nius Rohdaten — Topic Word Scores (Top 12)


In [138]:
if 'nius_rohdaten_topic_model' not in globals():
    raise RuntimeError('Run the corresponding source topic-modelling cell first.')
display(nius_rohdaten_topic_model.visualize_barchart(top_n_topics=12))


### Tichy's Einblick — Topic Word Scores (Top 12)


In [139]:
if 'tichys_einblick_topic_model' not in globals():
    raise RuntimeError('Run the corresponding source topic-modelling cell first.')
display(tichys_einblick_topic_model.visualize_barchart(top_n_topics=12))


### Reitschuster — Topic Word Scores (Top 12)


In [140]:
if 'reitschuster_topic_model' not in globals():
    raise RuntimeError('Run the corresponding source topic-modelling cell first.')
display(reitschuster_topic_model.visualize_barchart(top_n_topics=12))
